# Ein dichtes neuronales Netz erstellen
In diesem Notebook erstellst du dein eigenes dichtes neuronales Netz für die Vorhersage von 2 Klassen und implementierst dabei
1. Modellarchitektur
2. Initialisierung der Parameter
3. Vorwärtspropagierung
4. Berechnung der Verlustfunktion
5. Rückwärtspropagierung
6. Aktualisierung der Parameter per Gradientenabstieg
7. Trainingsalgorithmus
8. Vorhersage

Am Ende dieses Notebooks solltest du Folgendes gelernt haben:
- ein Verständnis davon, wie ein neuronales Netz aufgebaut ist
- wie ein neuronales Netz zu einer Vorhersage kommt
- wie Rückwärtspropagierung und Gradientenabstieg ein neuronales Netz trainieren
- wie die Dimension der verborgenen Schicht und die Lernrate das Ergebnis des Trainings und/oder der Vorhersage beeinflussen
- wie die Problemstellung die Leistung eines neuronalen Netzes bestimmt

## Python-Module importieren

In [ ]:
# Benötigte Pakete importieren
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

from sklearn import datasets

In [ ]:
## Datensatz laden
X, Y = datasets.make_moons(10000, noise=0.07, random_state=24)
plt.scatter(X[:, 0], X[:, 1], c=Y)
plt.title("The moon data")
plt.show();

Im Plot oben ist deutlich zu sehen, dass sich die Klassen dieses Datensatzes nicht durch eine lineare Entscheidungsgrenze trennen lassen. Eine logistische Regression würde hier also ziemlich schlecht abschneiden (probier es aus). 

Sehen wir uns die Dimensionen der Daten an.

In [ ]:
# Dimensionen der Daten ausgeben
print(" input dimensions: {}".format(X.shape))
print("output dimensions: {}".format(Y.shape))

## Voraussetzung: logistische Regression
Als geeignetes Benchmark-Modell sollte eine logistische Regression auf den Daten trainiert werden. Beachte, dass eine logistische Regression eine lineare Entscheidungsgrenze hat. Für die oben gezeigten Daten ist das problematisch. 

>__Aufgabe__: Trainiere ein logistisches Regressionsmodell auf den Daten und gib den Accuracy-Score auf dem Testdatensatz aus. Verwende eine 70/30-Aufteilung in Trainings- und Testdaten mit dem Random Seed 42. Trainiere das Modell mit dem bereits importierten `LogisticRegressionCV()`. Visualisiere die Entscheidungsgrenze mit der Funktion `visualize_decision_boundary()` aus `dnn_utils.py`. Beachte, dass `visualize_decision_boundary()` die `predict()`-Funktion eines Modells erwartet. Sie lässt sich mit einer `lambda`-Funktion übergeben: `lambda x: model.predict(x)`. 

In [ ]:
# ===========#
# DEIN CODE # ~ 4 Zeilen
# ===========#

## 1. Modellarchitektur
Für jede Schicht wird die Anzahl der Parameter jeweils durch ihre Eingabe und Ausgabe festgelegt. Das dichte neuronale Netz hat die folgenden Schichten:

1. Eingabeschicht
2. verborgene Schicht
3. Ausgabeschicht

<img src="../assets/artificial_network_architecture_dnn.png" alt="" width="400"/>


Alle Schichten haben eine Eingabe und eine Ausgabe. Die Eingabe der Eingabeschicht ist die Anzahl der Features `n_input` in `X`, und die Ausgabe der Ausgabeschicht ist die Anzahl der Klassen `n_output`. Bei einer Ausgabe mit 2 Klassen brauchen wir nur eine einzige Ausgabe, also `n_output=1`. Interessant ist das, was in der verborgenen Schicht passiert. Die Eingabe dieser Schicht ergibt sich aus der Ausgabe der vorherigen Schicht, der Eingabeschicht. Die Größe der Ausgabe der Eingabeschicht ist `n_input` (denk daran: Das ist eine Durchgangsschicht; sie lässt einfach alles durch). Die Ausgabedimension der verborgenen Schicht ist in der Variablen `n_hidden` festgelegt. Zusammen bestimmen diese Variablen die Größe der Gewichtsmatrizen und Biases, die das Netz lernen muss.

>__Aufgabe__: Definiere eine Funktion, die die Dimensionen der Schichten bestimmt.

In [ ]:
def model_architecture(X, Y, n_hidden=10):
    """
    Definiert die Architektur des DNN-Modells

        Args:
        X: Feature-Vektor der Form (n_samples, n_input)
        Y: tatsächliche Zielwerte der Form (n_samples, 1)
        n_hidden: Größe der verborgenen Schicht (benutzerdefiniert)

        Returns:
        arch: Modellarchitektur mit den Größen der Schichten in einem dict
    """

    # Größen der Schichten festlegen
    # ===========#
    # DEIN CODE # ~ 3 Zeilen
    # ===========#

    # ===========#
    # FERTIG    #
    # ===========#
    arch = {"n_input", "n_hidden", "n_output"}

    return arch

## 2. Initialisierung der Parameter
Die Initialisierung der Parameter ist bei neuronalen Netzen ein heikles Thema, denn anders als bei anderen Machine-Learning-Methoden können wir die Startgewichte nicht auf null setzen (für den Bias funktioniert das allerdings). Der Grund dafür: Ein Start bei null liefert nichtssagende Gradienten, und dein Modell verbessert sich nicht. 

In der Literatur zu neuronalen Netzen gibt es viele verschiedene Arten, die Parameter zu initialisieren. Manche davon werden häufiger verwendet als andere. Hier verwenden wir die zufällige Initialisierung der Gewichte, d. h., wir initialisieren alle Gewichte zufällig mit dem Mittelwert null und einer bestimmten Varianz (ist sie zu niedrig, neigen die Gradienten wieder dazu, zu verschwinden). 

**_Info_**: Zwei weitere erwähnenswerte Initialisierungen sind die _He-Initialisierung_ und die _Xavier-Initialisierung_, auch _Glorot-Initialisierung_ genannt (vor allem in mehreren Softwarepaketen).

Aus dem mathematischen Modell sollte klar werden, welche Parameter existieren und initialisiert werden müssen:

$$
\begin{align}
\mathbf{z}^{[1]}_i&=\mathbf{W}^{[1]}\mathbf{x}_i+\mathbf{b}^{[1]}\\
\mathbf{a}^{[1]}_i&=\tanh(\mathbf{z}^{[1]}_i)\\
\mathbf{z}^{[2]}_i&=\mathbf{W}^{[2]}\mathbf{a}^{[1]}_i+\mathbf{b}^{[2]}\\
\mathbf{a}^{[2]}_i&=\text{sigmoid}(\mathbf{z}^{[2]}_i)
\end{align}
$$

Ein Schwellenwert bestimmt dann, welche Klasse der $i$-ten Beobachtung zugeordnet wird:

$$
\hat{y}_i=\begin{cases}
1&a^{[2]}_i>0.5\\
0&a^{[2]}_i\leq0.5\\
\end{cases}
$$

**_Info_**: Neuronale Netze jeder Art werden üblicherweise so erstellt: Zuerst wird die Netzarchitektur festgelegt (d. h. `n_inputs`, `n_hidden` und `n_output`). Dann folgen Schritt (2.) von [oben](#2-initialisierung-der-parameter) (Initialisierung der Parameter) und eine Schleife über die Schritte (3.)-(5.). In vielen Programmen findest du eine umhüllende Funktion, die die Schritte  (1.)-(5.) enthält und dein ganzes Netz trainiert. Nach dem Training kannst du die trainierten Modellparameter für Vorhersagen verwenden.

>__Aufgabe__: Vervollständige die Funktion `init_params` zur Initialisierung der Parameter mit zufälligen Startparametern.

In [ ]:
def init_params(n_input, n_hidden, n_output):
    """
    Args:
    n_input:  Größe der Eingabeschicht
    n_hidden: Größe der verborgenen Schicht
    n_output: Größe der Ausgabeschicht

    Returns:
    params: dict mit den Parametern des Netzes:
        W1: Gewichtsmatrix der verborgenen Schicht
        b1: Bias der verborgenen Schicht
        W2: Gewichtsmatrix der Ausgabeschicht
        b2: Bias der Ausgabeschicht
    """
    # Der Seed dient zur Prüfung der Ausgaben
    np.random.seed(42)

    # ===========#
    # DEIN CODE #
    # ===========#

    assert W1.shape == (n_hidden, n_input)
    assert b1.shape == (n_hidden, 1)
    assert W2.shape == (n_output, n_hidden)
    assert b2.shape == (n_output, 1)

    # ===========#
    # FERTIG    #
    # ===========#
    params = {"W1", "b1", "W2", "b2"}

    return params

## 3. Vorwärtspropagierung
Die Vorwärtspropagierung ist der Schritt durch das Netzmodell von der Eingabe zur Ausgabe. 

>__Aufgabe__: Vervollständige die Funktion `forward_prop()` unten, um den Vorwärtsschritt durch das Netz zu berechnen. Du kannst die Funktion `sigmoid()` aus `dnn_utils.py` verwenden. 

**_Info_**: Die Funktion `forward_prop()` gibt neben der Modellausgabe einen `cache` zurück, der die Vektoren der Schichten für den Schritt der Rückwärtspropagierung enthält. Weil diese Werte zwischengespeichert werden, muss der Schritt der Rückwärtspropagierung sie nicht neu berechnen. Das spart Rechenzeit. 

In [ ]:
def forward_prop(X, params):
    """
    Args:
    X:      Eingabedaten (NxD)
    params: Modellparameter als Dictionary

    Returns:
    A:     Aktivierungen der Ausgabeschicht
    cache: dict mit den Aktivierungen A und der Schichtausgabe Z
    """

    # Vorwärtspropagierung

    # ===========#
    # DEIN CODE # ~4 Zeilen
    # ===========#

    assert A2.shape == (1, X.shape[0])

    # ===========#
    # FERTIG    #
    # ===========#
    cache = {"Z1", "A1", "Z2", "A2"}

    return A2, cache

## Berechnung der Verlustfunktion
Nach der Vorwärtspropagierung wird die Ausgabe des Netzes in der Verlustfunktion verwendet, um die Ausdrucksstärke des Modells zu evaluieren. Hier verwenden wir die _Log-Loss_-Funktion (das ist eigentlich die Form der _Kreuzentropie_ für 2 Klassen):

$$
\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})=-\frac{1}{N}\sum_{i=1}^N\left(y_i\log(a^{[2]}_i)+(1-y_i)\log(1-a^{[2]}_i)\right)
$$

wobei $N$ die Anzahl der Beobachtungen `n_samples` ist und $a_i^{[2]}$ das $i$-te Element der Aktivierung der Ausgabeschicht, $\mathbf{a}^{[2]}$, bezeichnet.

**_Info_**: Mit Matrixmultiplikation lässt sich die Berechnung des Verlusts vektorisieren, das erhöht die Geschwindigkeit (verwende keine Schleife). Sieh dir `np.multiply()` an. Denk daran, dass die Verlustfunktion am Ende einen skalaren Wert zurückgibt. 

>__Aufgabe__: Vervollständige die Funktion `logloss()` unten, um die Verlustfunktion des Modells zu berechnen, die bei der Optimierung verwendet wird. 

In [ ]:
def logloss(A, Y):
    """
    Berechnet den Log-Loss (Kreuzentropie)

        Arguments:
        A:      Aktivierungen der Ausgabe
        Y:      tatsächliche Zielwerte der Form (n_samples,)

        Returns:
        logloss: Kosten der aktuellen Parameterwahl
    """

    # Verlustfunktion hier implementieren
    # ===========#
    # DEIN CODE # ~2 Zeilen
    # ===========#

    # Dimensionen des skalarwertigen Vektors reduzieren
    loss = float(np.squeeze(loss))

    return loss

## Rückwärtspropagierung 
Die Rückwärtspropagierung propagiert den Fehler mithilfe der Kettenregel für Ableitungen zurück in das Netz. Gebraucht werden die Ableitungen der Verlustfunktion nach jedem Parameter, d. h. nach allen Gewichten und Biases. Die Rückwärtspropagierung ist meist der anspruchsvollste Teil beim Programmieren eines Deep-Learning-Netzes. 

Die folgenden Ableitungen müssen implementiert werden: 

$$
\begin{align}
\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial z^{[2]}_i}&=(a^{[2]}_i-y_i)\\
\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial\mathbf{W}^{[2]}}&=\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial z^{[2]}_i}\mathbf{a}^{[1]T}_i\\
\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial b^{[2]}}&=\sum_{i=1}^N\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial z^{[2]}_i}\\
\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial \mathbf{z}^{[1]}_i}&=\mathbf{W}^{[2]T}\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial z^{[2]}_i}*(\mathbf{1}-(\mathbf{a}^{[1]}_i)^2)\\
\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial \mathbf{W}^{[1]}}&=\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial \mathbf{z}^{[1]}_i}\mathbf{x}_i^T\\
\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial \mathbf{b}^{[1]}}&=\sum_{i=1}^N\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial\mathbf{z}^{[1]}_i}
\end{align}
$$
wobei $T$ für die Transponierte eines Vektors bzw. einer Matrix steht und $\mathbf{1}$ ein Vektor aus Einsen ist. Der $*$-Operator bezeichnet eine Faltung, d. h. eine elementweise Multiplikation für Vektoren und Matrizen.

**_Info_**: Dir fällt vielleicht auf, dass du auch die Aktivierungsfunktionen $\phi()=\{\tanh,\text{sigmoid}\}$ ableiten musst, um von den Aktivierungen zurück zu den Parametern zu gelangen. Für $\phi=\tanh$ gilt $\phi^{\prime}(\mathbf{z}^{[1]})=1-(\mathbf{a}^{[1]})^2$. 

>__Aufgabe__: Vervollständige die Funktion `backward_prop()` unten und setz dabei all dein Wissen über Matrixalgebra ein. Normalisiere alle Gradienten der Parameter mit der Anzahl der Beobachtungen, um glattere Batch-Gradienten zu erhalten. (_Tipp_: Die Ableitung der Verlustfunktion nach der Ausgabe der zweiten Schicht, $\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial z^{[2]}_i}=(a^{[2]}_i-y_i)$, ist in Vektorform $A2-Y$)

In [ ]:
def backward_prop(X, Y, params, cache):
    """
    Args:
    X:      Feature-Matrix der Form (n_samples, n_inputs)
    Y:      tatsächliche Zielwerte der Form (n_samples,1)
    params: Modellparameter als Dictionary
    cache:  zwischengespeicherte Ausgaben der Schichten (Z1,A1,Z2,A2)

    Returns:
    grads: dict mit den Gradienten
    """

    # Gradienten der Parameter mit dieser Batchgröße normalisieren.
    n_samples = X.shape[0]

    # Rückwärtspropagierung:
    # Berechnen: dW2, db2, dW1, db1
    # Das sind sechs Gleichungen.
    # ===========#
    # DEIN CODE # ~6 Zeilen
    # ===========#

    grads = {"dW1", "db1", "dW2", "db2"}

    return grads

## 6. Aktualisierung der Parameter per Gradientenabstieg
Die Aktualisierung der Parameter beim Gradientenabstieg für neuronale Netze ähnelt eigentlich der allgemeinen: 

$$
\begin{align}
\mathbf{W}^{[j]}_{t+1}&=\mathbf{W}^{[j]}_t - \alpha\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial\mathbf{W}^{[j]}}\big|_t\\
\mathbf{b}^{[j]}_{t+1}&=\mathbf{b}^{[j]}_{t} - \alpha\frac{\partial\mathcal{L}(\mathbf{y},\mathbf{a}^{[2]})}{\partial\mathbf{b}^{[j]}}\big|_t
\end{align}
$$

wobei $\alpha$ die Lernrate ist (Schrittweite des Gradientenabstiegs). Eine zu kleine Lernrate verlängert die Trainingszeit, und eine zu große Lernrate lässt den Algorithmus divergieren. Die Lernrate ist also ein Hyperparameter, der Sorgfalt verlangt. Sie ist einer der wichtigsten Hyperparameter bei der Optimierung neuronaler Netze. Theoretisch muss die Lernrate ausreichend schnell sinken, damit der Gradientenabstieg sicher konvergiert.  


>__Aufgabe__: Vervollständige die Funktion `update_params()` unten, um die Parameter per Gradientenabstieg zu aktualisieren.

In [ ]:
def update_params(params, grads, learning_rate=1.5):
    """
    Args:
    params:        Modellparameter als Dictionary
    grads:         Gradienten der Parameter als Dictionary
    learning_rate: Schrittweite des Gradientenabstiegs

    Returns:
    params: aktualisierte Modellparameter als Dictionary
    """

    # Alle Parameter aktualisieren
    # ===========#
    # DEIN CODE # ~4 Zeilen
    # ===========#

    params = {"W1", "b1", "w2", "b2"}

    return params

## 7. Trainingsalgorithmus
Der Trainingsalgorithmus fügt alle Schritte zusammen. Hier erstellst du dein komplettes neuronales Netz. Der Gradientenabstieg wird hier in _Batch_-Form verwendet, d. h., wir verwenden bei jeder Aktualisierung __alle__ Beobachtungen im Datensatz. Die maximale Anzahl der Iterationen legt fest, wie oft aktualisiert wird. 

>__Aufgabe__: Vervollständige die Funktion `train_dnn()` unten, um das Training für dein neuronales Netz zu implementieren.

In [ ]:
def train_dnn(X, Y, n_hidden, max_iter=10000, learning_rate=1.2, verbose=False):
    """
    Trainiert ein dichtes neuronales Netz mit einer verborgenen Schicht aus 'n_hidden' Neuronen.

        Args:
        X:             Feature-Vektor der Form (n_samples,n_input)
        Y:             tatsächliche Zielwerte der Form (n_samples,1)
        n_hidden:      Anzahl der Neuronen in der verborgenen Schicht
        learning_rate: Schrittweite des Gradientenabstiegs
        max_iter:      maximale Anzahl der Iterationen des Gradientenabstiegs
        verbose:       Infos während der Optimierung ausgeben

        Returns:
        params: dict mit den trainierten Modellparametern (für Vorhersagen verwendbar)
    """

    # Parameter des Modells initialisieren
    # ===========#
    # DEIN CODE # ~1 Zeile
    # ===========#

    # Gradientenabstieg
    # Iterieren, bis der Iterator erschöpft ist
    for i in range(max_iter):
        # 1. Vorwärtspropagierung
        # ===========#
        # DEIN CODE # ~1 Zeile
        # ===========#

        # 2. Verlust berechnen
        # ===========#
        # DEIN CODE # ~1 Zeile
        # ===========#

        # 3. Rückwärtspropagierung
        # ===========#
        # DEIN CODE # ~1 Zeile
        # ===========#

        # 4. Aktualisierung der Parameter
        # ===========#
        # DEIN CODE # ~1 Zeile
        # ===========#

        # Infos zum Gradientenabstieg
        if verbose:
            if i % 500 == 0:
                print("Loss after %i iterations: %f" % (i + 1, loss))

    return params

## 8. Vorhersage
Bei der Vorhersage werden die trainierten Parameter verwendet, um eine Ausgabe des Netzes zu berechnen, und ein Schwellenwert ordnet die Klasse zu. Achte darauf, eine Funktion wiederzuverwenden, die du bereits programmiert hast. 

>__Aufgabe__: Vervollständige die Funktion `predict()` für dein neuronales Netz. Verwende einen Schwellenwert von 0,5.

In [ ]:
def predict(X, params):
    """
    Sagt Zielwerte mit dem trainierten Modell vorher

        Args:
        X: Feature-Vektor der Form (n_samples, n_input)
        params: Modellparameter als Dictionary

        Returns:
        preds: Vorhersagen des Modells
    """

    # Das DNN gibt Wahrscheinlichkeiten aus
    # Ein Schwellenwert legt fest, wie die Ergebnisse den Klassen zugeordnet werden
    # ===========#
    # DEIN CODE # ~2 Zeilen
    # ===========#

    return preds

In [ ]:
# Werte vorhersagen
predicted = predict(X_train, params)

print("Predictions: ")
print("------------")
print("Train accuracy: %f" % accuracy_score(y_train, predicted))

## Dein neuronales Netz auf den Daten trainieren

>__Aufgabe__: Trainiere das dichte neuronale Netz auf den oben gegebenen Daten und erstell Vorhersagen. Verwende eine 70/30-Aufteilung in Trainings- und Testdaten und gib die Test-Accuracy aus. Nutz die Funktion `visualize_decision_boundary()` aus `dnn_utils.py`, um die Entscheidungsgrenze des Netzes zu visualisieren. 

In [ ]:
# ===========#
# DEIN CODE # ~3 Zeilen
# ===========#
print("Training results: ")
print("-----------------")
print("train acc.: %f" % accuracy_score(y_train, preds_train))
print(" test acc.: %f" % accuracy_score(y_test, preds_test))

### Dein neuronales Netz tunen

>__Aufgabe__: Tune dein Netz aus der letzten Aufgabe, indem du die Dimension der verborgenen Schicht änderst. Verwende eine 80/20-Aufteilung des Trainingsdatensatzes aus der letzten Aufgabe in Trainings- und Validierungsdaten und gib für jede Wahl des Hyperparameters die Test-Accuracy aus. Nutz die Funktion `visualize_decision_boundary()` aus `dnn_utils.py`, um zu visualisieren, wie gut dein Netz wird.

In [ ]:
# ===========#
# DEIN CODE # ~10 Zeilen
# ===========#

### Training des neuronalen Netzes verbessern

>__Zusatzaufgabe__: Spiel mit der _Lernrate_ des Gradientenabstiegs herum. Was beobachtest du? 

In [ ]:
# ===========#
# DEIN CODE #
# ===========#

## Weitere Datensätze (optional)

Der Gradient der Gewichte ergibt sich auch direkt aus dem Fehler über das äußere Produkt:>__Zusatzaufgabe__: Probier die unten aufgeführten weiteren Datensätze aus und untersuche, wie dein dichtes neuronales Netz auf diesen Daten abschneidet (dafür musst du das Netz neu trainieren). 

Datensätze:
- `sklearn.datasets.make_blobs()`: Denk daran, dass dein Netz nur mit 2-dimensionalen Features und zwei Klassen umgehen kann
- `sklearn.datasets.make_circles()`
- `sklearn.datasets.make_classifications()` 
- `sklearn.datasets.make_hastie_10_2()` 
- `sklearn.datasets.make_gaussian_quantiles()` 
- du kannst auch ausprobieren, was passiert, wenn du Zufallsdaten ganz ohne Struktur (aber mit zwei Klassen) verwendest 

In [ ]:
# ===========#
# DEIN CODE #
# ===========#